In [2]:
#skip this cell.
#This was for library development purposes only.
import sys
from pathlib import Path

src_path = Path("src")

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


In [3]:
import PyVisualFields
print(PyVisualFields.__file__)
print(PyVisualFields.__version__)

/home/eslamim/Partners HealthCare Dropbox/Mohammad Eslami/GithubRepos/PyVisualField/src/PyVisualFields/__init__.py
2.0.4


In [4]:
from PyVisualFields import visualFields
from PyVisualFields.utils import canonicalize_vf_df
from PyVisualFields.utils import vf_blocks, missing_blocks
from PyVisualFields.utils import compute_missing_blocks
from PyVisualFields.utils import print_vf_summary, investigate_vf_df
from PyVisualFields import vfprogression
import pandas as pd

df_VFs = canonicalize_vf_df(vfprogression.data_vfseries())

print('---> blocks',vf_blocks(df_VFs))
print('---> missed blocks', missing_blocks(df_VFs))

#### now you can investigate the available information in the dataframe
print_vf_summary(df_VFs) # this function will print

---> blocks {'sens': True, 'td': True, 'pd': True, 'tdp': True, 'pdp': True}
---> missed blocks []
=== VF Data Summary ===
Rows: 20
Columns: 286

Pointwise blocks:
  ✓ sensitivity (54 locations)
  ✓ total_deviation (54 locations)
  ✓ pattern_deviation (54 locations)
  ✓ total_deviation_probability (54 locations)
  ✓ pattern_deviation_probability (54 locations)

Global indices:
  ✓ md
  ✓ psd
  ✓ ght
  ✓ vfi
  ✗ msens
  ✗ ssens
  ✗ tmd
  ✗ tsd
  ✗ pmd
  ✗ gh

Metadata:
  ✗ patientid
  ✓ eyeid
  ✗ date
  ✓ age
  ✓ yearsfollowed
  ✓ duration


In [5]:

# glaucoma metrics depends on TD, Tdp, PD and PDP values.
# also GHT, PSD, MD, etc.
# so let's compute them if not available 
# GHT can not be calculated
import pandas as pd
from PyVisualFields.utils import compute_missing_blocks
from PyVisualFields import visualFields

print(df_VFs.shape)
df_VFs = compute_missing_blocks(df_VFs)
print('---> blocks',vf_blocks(df_VFs))
print('---> missed blocks', missing_blocks(df_VFs))
print(df_VFs.shape)

df_gi = visualFields.getgl(df_VFs) 
combined_df = pd.concat([df_VFs, df_gi], axis=1)
final_df = combined_df.loc[:, ~combined_df.columns.duplicated()]

(20, 286)
---> blocks {'sens': True, 'td': True, 'pd': True, 'tdp': True, 'pdp': True}
---> missed blocks []
(20, 286)
==> py_getgl: missing global indices to compute: ['msens', 'ssens', 'tmd', 'tsd', 'pmd', 'gh']


In [6]:
final_df.head()

,eyeid,nvisit,yearsfollowed,distprev,age,righteye,duration,falsenegrate,falseposrate,malfixrate,...,pdp51,pdp52,pdp53,pdp54,msens,ssens,tmd,tsd,pmd,gh
0,1.0,1,0.00,NaN,64.98,1,287,0.0,0.00,0.071429,...,1.0,1.0,1.0,1.0,29.000000,2.283788,-0.621837,1.537917,-1.854798,1.0
1,1.0,2,0.73,0.73,65.71,1,295,0.0,0.01,0.142857,...,1.0,1.0,1.0,1.0,29.346154,2.383642,-0.309503,1.574159,-1.489677,1.0
2,1.0,3,1.77,1.04,66.75,1,291,0.0,0.01,0.062500,...,1.0,1.0,1.0,1.0,29.461538,2.287747,-0.189323,1.899977,-1.968863,2.0
3,1.0,4,2.78,1.01,67.76,1,347,0.0,0.02,0.066667,...,1.0,1.0,1.0,1.0,27.576923,2.538690,-1.843116,1.862786,-1.803049,0.0
4,1.0,5,4.01,1.23,68.99,1,309,0.0,0.01,0.000000,...,1.0,1.0,1.0,1.0,27.500000,2.866302,-1.986323,2.610211,-2.922875,1.0


In [7]:
from PyVisualFields import PyGlaucoMetrics 

In [8]:
""" ght column can be integer coded or string coded. 
    1: "Within Normal Limits",
    2: "Borderline",    
    3: "Outside Normal Limits",    
    4: "General Reduction of Sensitivity",
    5: "Abnormally High Sensitivity",
    6: Borderline/General Reduction
"""

# The demo dataset is integer-coded

' ght column can be integer coded or string coded. \n    1: "Within Normal Limits",\n    2: "Borderline",    \n    3: "Outside Normal Limits",    \n    4: "General Reduction of Sensitivity",\n    5: "Abnormally High Sensitivity",\n    6: Borderline/General Reduction\n'

# HAP2 criteria
Pattern Deviation Probabilities


GHT “outside normal limits” 

or

cluster of 3 points at P < 5%, 1 of which must be at P < 1% (if the PD plot was not reported due to
severe general field depression, then the cluster was considered present)

or

PSD at P < 5%

In [9]:
# classify the visual fields for glaucoma based on HAP2
# Determined class will be shown correspondingly in columns: 'HAP2_clf'

# Hap2 criteria can consider the psd and GHT too. but optional
# so we optionally added them, but nan for this example
# Hap2 PartI: Minimum criteria for diagnosing acquired glaucomatous damage

df_diag_HAP2 = PyGlaucoMetrics.Fn_HAP2(final_df)
print(df_diag_HAP2[['l1', 'l2', 'HAP2_clf']].head())

   l1  l2 HAP2_clf
0  26  28   Non-GL
1  26  25   Non-GL
2  28  27   Non-GL
3  24  25       GL
4  26  26       GL


In [10]:
## HAP 2 part II: classification of the severity of visual field defects
# Part II – Classification of glaucomatous visual field defects
# It needs sensitvity, and pdp, and MD
# the severity results are shown in severity(HAP2_part2)
# Early, Moderate, Severe
# https://optometricglaucomasociety.org/wp-content/uploads/2024/01/Clinical-Decisions-in-Glaucoma_compressed.pdf
import pandas as pd
df_diag_HAP2withPartII = PyGlaucoMetrics.Fn_HAP2_part2(df_diag_HAP2)
print(df_diag_HAP2withPartII[['l1', 'l2', 'HAP2_clf', 'severity(HAP2_part2)']].head())

   l1  l2 HAP2_clf severity(HAP2_part2)
0  26  28   Non-GL                Early
1  26  25   Non-GL                Early
2  28  27   Non-GL                Early
3  24  25       GL                Early
4  26  26       GL                Early


# UKGTS criteria
Total Deviation and Total Deviation Probabilities


cluster of 2 points at P < 1% 

or

cluster of 3 points at P < 5%

or

cluster of 2 points with 10 dB difference across the nasal horizontal midline

In [11]:
# classify the visual fields for glaucoma based on HAP2
# Determined class will be shown correspondingly in columns: 'UKGTS_clf',

df_diag_UKGTS = PyGlaucoMetrics.Fn_UKGTS(final_df)
print(df_diag_UKGTS[['l1', 'l2', 'UKGTS_clf']].head())

   l1  l2 UKGTS_clf
0  26  28    Non-GL
1  26  25    Non-GL
2  28  27    Non-GL
3  24  25        GL
4  26  26        GL


# LoGTS criteria
Total Deviation

cluster of 2 points at < −10 dB
or
cluster of 3 points at < −8 dB

In [12]:
# classify the visual fields for glaucoma based on LoGTS
# Determined class will be shown correspondingly in columns: 'LoGTS_clf',
df_diag_LoGTS = PyGlaucoMetrics.Fn_LoGTS(final_df)
print(df_diag_LoGTS[['l1', 'l2', 'LoGTS_clf']].head())

   l1  l2 LoGTS_clf
0  26  28    Non-GL
1  26  25    Non-GL
2  28  27    Non-GL
3  24  25    Non-GL
4  26  26    Non-GL


# Foster criteria:
    GHT outside normal limits
    AND
    cluster of 3 points at P < 5%
    (if PD unavailable due to severe depression,
     cluster criterion automatically satisfied)

In [13]:
# classify the visual fields for glaucoma based on Foster
# Determined class will be shown correspondingly in columns: 'Foster_clf',

# Foster needs GHT, so if it is not available, you will receive an error

df_diag_Foster = PyGlaucoMetrics.Fn_Foster(final_df)
print(df_diag_Foster[['l1', 'l2', 'Foster_clf']].head())

   l1  l2 Foster_clf
0  26  28     Non-GL
1  26  25     Non-GL
2  28  27     Non-GL
3  24  25         GL
4  26  26         GL



# Kangs criteria
Total Deviation

criterion 1: cluster of 3 contiguous points with TD < -5 dB 

    or 
    
criterion 2: cluster of 2 contiguous points with TD < -10 dB

    

In [14]:
# classify the visual fields for glaucoma based on Kang
# Determined class will be shown correspondingly in columns:  'Kangs_clf'
df_diag_Kangs = PyGlaucoMetrics.Fn_Kangs(final_df)
print(df_diag_Kangs[['l1', 'l2', 'Kangs_clf']].head())

   l1  l2 Kangs_clf
0  26  28    Non-GL
1  26  25    Non-GL
2  28  27    Non-GL
3  24  25    Non-GL
4  26  26        GL
